# House Prices (Ames)

* Data source: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data
* Team:
* Team members:

**Dataset information**
| Column | Meaning |
|---|---|
| `SalePrice` | Price the house sold for (USD). This is what we predict. |
| `GrLivArea` | Living area above ground (sq ft) |
| `TotalBsmtSF`, `1stFlrSF`, `2ndFlrSF` | Basement area, 1st floor area, 2nd floor area (sq ft) |
| `LotArea`, `LotFrontage` | Size of the lot (sq ft), length of street touching the lot (ft) |
| `OverallQual`, `OverallCond` | Overall build quality and overall condition, scored 1 (very poor) up to 10 (very excellent) |
| `ExterQual`, `KitchenQual`, `BsmtQual`, ... | Quality ratings written as letters (`Po`, `Fa`, `TA`, `Gd`, `Ex`). Shown in section 1.4. |
| `Neighborhood` | Which part of Ames the house is in (25 areas) |
| `MSSubClass` | A code for the type of house (one-story, two-story, duplex...). Shown in section 1.4. |
| `SaleCondition` | How the sale happened (normal, foreclosure, new house not finished...) |
| `YearBuilt`, `YearRemodAdd`, `YrSold`, `MoSold` | Year built, year remodelled, year sold, month sold |
| `NA` in a column like `GarageType` | Usually means the house has no garage (or basement, pool...). Explained in section 1.2. |

## 0. SET UP

Download all necessary libraries

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn

Unzip dataset (skipped if there is no zip file and `train.csv` is already there)

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("dataset/house.zip")
extract_dir = Path("dataset/unzip")
extract_dir.mkdir(parents=True, exist_ok=True)

# Google Colab: upload house.zip (or train.csv) if it is not there yet.
try:
    from google.colab import files  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not zip_path.exists() and not (extract_dir / "train.csv").exists():
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    print("Upload dataset/house.zip (Kaggle zip) or train.csv:")
    for name, data in files.upload().items():
        target = zip_path if name.endswith(".zip") else extract_dir / name
        target.write_bytes(data)

if zip_path.exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)

print("Files:", sorted(p.name for p in extract_dir.iterdir()))

Imports, settings, load the data.
> In this file, `NA` means feature absent, and `None` is a category value. They are not missing data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.max_columns", 50)

raw = pd.read_csv("dataset/unzip/train.csv", keep_default_na=False, na_values=["NA"])
print("Rows, columns:", raw.shape)
raw.head(3)

## 1. DATA CLEANING

**Goal**: find the problems in the raw data and decide what to do about each one.

**Rule**: we only use fixed rules (what a column means, what is obviously wrong).


### 1.1. Basic overview

In [ ]:
df = raw.drop(columns="Id")

print("Rows, columns:", df.shape)
print()
print("Column types:")
print(df.dtypes.astype(str).value_counts().to_string())

* 1,460 houses and 80 columns (`Id` dropped). 43 are text, 34 are whole numbers, 3 have decimals.

In [ ]:
look = ["SalePrice", "LotArea", "LotFrontage", "MasVnrArea", "MiscVal"]
df[look].describe().loc[["count", "mean", "50%", "75%", "max"]]

* `SalePrice`: the mean (180,921) is higher than the median (163,000), and the max (755,000) is far above the 75% mark (214,000). A few very expensive houses stretch the price to the right.
* Some columns have a max far away from it's 75%:
    * `LotArea` max 215,245 vs 75% = 11,601
    * `LotFrontage` max 313 vs 75% = 80
    * `MasVnrArea` max 1,600 vs 75% = 166
    * `MiscVal` max 15,500 vs 75% = 0
* `LotFrontage` has 1,201 values out of 1,460, so 259 are missing.

### 1.2. Missing values

In [ ]:
# Columns where NA means "the house doesn't have it"
absent_cat_cols = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
                   "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond",
                   "PoolQC", "Fence", "MiscFeature"]
absent_num_cols = ["GarageYrBlt"]       # NA = no garage

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": (missing / len(df) * 100).round(1),
})
missing_report["meaning"] = np.where(missing_report.index.isin(absent_cat_cols + absent_num_cols),
                                     "house doesn't have it", "really missing")
print("Columns with NA:", len(missing_report))
display(missing_report)

colors = missing_report["meaning"].map({"house doesn't have it": "steelblue", "really missing": "crimson"})
plt.figure(figsize=(9, 6))
plt.barh(missing_report.index, missing_report["missing_pct"], color=colors)
plt.gca().invert_yaxis()
plt.xlabel("% of houses")
plt.title("Columns with NA (blue = house doesn't have it, red = really missing)")
plt.tight_layout()
plt.show()

- 19 columns have NA. In 15 of them (blue) NA means the house doesn't have it: `PoolQC` 99.5%, `MiscFeature` 96.3%, `Alley` 93.8%, `Fence` 80.8%, `FireplaceQu` 47.3%, the five garage columns 5.5%, the basement columns 2.5-2.6%.
- Only 4 columns are really missing (red): `LotFrontage` 259 houses (17.7%), `MasVnrType` 8, `MasVnrArea` 8 (the same 8 houses) and `Electrical` 1.

Is `NA` really "the house doesn't have it"? Here are three houses with `NA` in the garage columns, and three with `NA` in the basement columns:

In [ ]:
print("Houses with NA in GarageType:")
display(df.loc[df["GarageType"].isna(), ["GarageType", "GarageFinish", "GarageCars", "GarageArea"]].head(3))
print("Houses with NA in BsmtQual:")
display(df.loc[df["BsmtQual"].isna(), ["BsmtQual", "BsmtFinType1", "TotalBsmtSF"]].head(3))

- The example rows show it: houses with `NA` in `GarageType` have `GarageCars = 0` and `GarageArea = 0`. Houses with `NA` in `BsmtQual` have `TotalBsmtSF = 0`.

Now the same check for every group. A "mismatch" is a house where the `NA` and the size disagree (for example: `NA` for the garage but a garage area bigger than 0).


In [ ]:
consistency = pd.Series({
    "BsmtQual is NA  but  TotalBsmtSF > 0 (or the reverse)":   int((df["BsmtQual"].isna()    != (df["TotalBsmtSF"] == 0)).sum()),
    "GarageType is NA  but  GarageArea > 0 (or the reverse)":  int((df["GarageType"].isna()  != (df["GarageArea"] == 0)).sum()),
    "FireplaceQu is NA  but  Fireplaces > 0 (or the reverse)": int((df["FireplaceQu"].isna() != (df["Fireplaces"] == 0)).sum()),
    "PoolQC is NA  but  PoolArea > 0 (or the reverse)":        int((df["PoolQC"].isna()       != (df["PoolArea"] == 0)).sum()),
}, name="mismatches")
display(consistency.to_frame())

print("MasVnrType and MasVnrArea are NA on the same houses:", bool((df["MasVnrType"].isna() == df["MasVnrArea"].isna()).all()))

- The mismatch table is all zeros for garage, basement, fireplace and pool, so `NA` here means "none", not missing.

In [ ]:
has_bsmt = df["BsmtQual"].notna()
print("Houses WITH a basement but NA in BsmtExposure:", int((has_bsmt & df["BsmtExposure"].isna()).sum()))
print("Houses WITH a basement but NA in BsmtFinType2: ", int((has_bsmt & df["BsmtFinType2"].isna()).sum()))

- Small catch: `BsmtExposure` and `BsmtFinType2` have 38 NA while the other basement columns have 37. That is 1 house each that has a basement but no value. It is tiny, so we treat it like the rest.

**Action**:
- Fill the "none" columns with the word `None` (text columns) or `0` (`MasVnrArea`). This is a fixed rule, no learning from data.
- For the 8 houses with no veneer info, we also use `None` / `0`, since "no veneer" is the most common case.
- Leave `LotFrontage` and `Electrical` empty for now. Filling them needs numbers from the training set, so that happens in the pipeline.


### 1.3. Duplicates checks

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

* 0 duplicates -> nothing to remove

### 1.4. Columns that need a type decision
Some columns are stored as numbers or letters, but the model needs to know what kind of thing they are. We check four groups, with prices as evidence.


#### `MSSubClass`
`MSSubClass` is stored as a number, but it is a code for the type of house. Is it a real scale (bigger code = more of something), or just a label? We check the median price per code.

In [ ]:
mssub_names = {20: "1-story, built 1946 or later", 30: "1-story, built before 1946", 40: "1-story with finished attic",
               45: "1.5-story, unfinished", 50: "1.5-story, finished", 60: "2-story, built 1946 or later",
               70: "2-story, built before 1946", 75: "2.5-story", 80: "split or multi-level", 85: "split foyer",
               90: "duplex", 120: "1-story PUD, 1946 or later", 150: "1.5-story PUD",
               160: "2-story PUD, 1946 or later", 180: "PUD multi-level / split", 190: "2-family conversion"}

mssub = df.groupby("MSSubClass")["SalePrice"].agg(["count", "median"])
mssub.insert(0, "meaning", mssub.index.map(mssub_names))
display(mssub)

plt.figure(figsize=(10, 3.8))
plt.bar(mssub.index.astype(str), mssub["median"], color="steelblue")
plt.xlabel("MSSubClass code")
plt.ylabel("Median SalePrice")
plt.title("Median price per MSSubClass code")
plt.tight_layout()
plt.show()

#### `MoSold`
Mont sold (1-12). Does the price go up or down in a straight line with the month number?


In [ ]:
month_price = df.groupby("MoSold")["SalePrice"].median()
plt.figure(figsize=(8, 3.5))
plt.bar(month_price.index.astype(str), month_price.values, color="steelblue")
plt.xlabel("Month sold")
plt.ylabel("Median SalePrice")
plt.title("Median price per month sold")
plt.tight_layout()
plt.show()

#### `OverallQual` and `OverallCond`
`OverallQual` and `OverallCond` are scores. Does a higher score mean a higher price, step by step?


In [ ]:
pd.DataFrame({
    "OverallQual": df.groupby("OverallQual")["SalePrice"].median(),
    "OverallCond": df.groupby("OverallCond")["SalePrice"].median(),
}).T

#### The rating columns
 Ten text columns use letters. Here are all ten with every value they contain. Reading the letters from low to high: `Po` = poor, `Fa` = fair, `TA` = typical/average, `Gd` = good, `Ex` = excellent, and `NA` = the house doesn't have it (no basement, no garage...).

In [ ]:
rating_cols = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
               "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
scale_order = ["Po", "Fa", "TA", "Gd", "Ex"]

rows = []
for c in rating_cols:
    found = df[c].fillna("NA").unique().tolist()
    ordered = [v for v in scale_order if v in found] + (["NA"] if "NA" in found else [])
    rows.append({"column": c,
                 "values found (low to high)": ", ".join(ordered),
                 "only uses Po/Fa/TA/Gd/Ex/NA": set(found) <= set(scale_order + ["NA"]),
                 "houses with NA": int(df[c].isna().sum())})
pd.DataFrame(rows).set_index("column")

Do the letters really work as a scale? Median price for two of them, from low to high:

In [ ]:
for c in ["ExterQual", "KitchenQual"]:
    present = [v for v in scale_order if v in df[c].unique()]
    print(c)
    display(df.groupby(c)["SalePrice"].agg(["count", "median"]).reindex(present))

**Observed**
- `MSSubClass`: price does not follow the code. 60 is expensive, 70 drops back, 30 is cheap, 120 jumps up again. The codes are labels, not a scale.
- `MoSold`: price does not climb or fall with the month number. It moves up and down. Also, month 12 is next to month 1 in real life, but a plain number would put them far apart.
- `OverallQual`: price goes up at every step. That is a real scale. `OverallCond` does not go up at every step, so it is a weaker scale.
- All ten rating columns use only the same six values (`Po`, `Fa`, `TA`, `Gd`, `Ex`, `NA`).
- For `ExterQual` and `KitchenQual`, price rises with each step up the letters. So the order is real.

**Action**
- Turn `MSSubClass` and `MoSold` into text, so they are treated as labels.
- Keep `OverallQual` and `OverallCond` as numbers (the model can still use them).
- Turn the ten rating columns into numbers: `NA` = 0, `Po` = 1, `Fa` = 2, `TA` = 3, `Gd` = 4, `Ex` = 5. (`PoolQC` is dropped in 1.7, so nine get mapped.)


### 1.5. Do the numbers add up?

In [ ]:
checks = pd.Series({
    "GrLivArea is not 1stFlrSF + 2ndFlrSF + LowQualFinSF": int((df["GrLivArea"] != df["1stFlrSF"] + df["2ndFlrSF"] + df["LowQualFinSF"]).sum()),
    "TotalBsmtSF is not BsmtFinSF1 + BsmtFinSF2 + BsmtUnfSF": int((df["TotalBsmtSF"] != df["BsmtFinSF1"] + df["BsmtFinSF2"] + df["BsmtUnfSF"]).sum()),
}, name="houses that don't add up")
checks.to_frame()

**Observed**: both checks give 0, so the area columns add up exactly.

**Action**: it is safe to add area columns together later (for example `TotalSF` in 2.2).

### 1.6. Outliers

In [ ]:
OUT_AREA, OUT_PRICE = 4000, 300000
outlier_rule = (df["GrLivArea"] > OUT_AREA) & (df["SalePrice"] < OUT_PRICE)

plt.figure(figsize=(8, 5))
plt.scatter(df.loc[~outlier_rule, "GrLivArea"], df.loc[~outlier_rule, "SalePrice"], s=12, alpha=0.5, label="other houses")
plt.scatter(df.loc[outlier_rule, "GrLivArea"], df.loc[outlier_rule, "SalePrice"], s=60, color="red", label="flagged")
plt.axvline(OUT_AREA, color="gray", linestyle="--")
plt.axhline(OUT_PRICE, color="gray", linestyle="--")
plt.xlabel("GrLivArea (sq ft)")
plt.ylabel("SalePrice")
plt.title("Living area vs price")
plt.legend()
plt.tight_layout()
plt.show()

print("Houses flagged:", int(outlier_rule.sum()))
display(df.loc[outlier_rule, ["GrLivArea", "SalePrice", "OverallQual", "Neighborhood", "SaleCondition"]])


Does removing these two houses change the link between area and price? (correlation between `GrLivArea` and log price)

In [ ]:
log_price_all = np.log1p(df["SalePrice"])
r_with = df["GrLivArea"].corr(log_price_all)
r_without = df.loc[~outlier_rule, "GrLivArea"].corr(log_price_all[~outlier_rule])
print(f"With the 2 houses:    {r_with:.3f}")
print(f"Without the 2 houses: {r_without:.3f}")

In 1.1 we also saw other big values (`LotArea`, `LotFrontage`, `MasVnrArea`, `TotalBsmtSF`). Here is the house with the biggest value of each, and its price:

In [ ]:
rows = []
for c in ["LotArea", "LotFrontage", "MasVnrArea", "TotalBsmtSF"]:
    i = df[c].idxmax()
    rows.append({"column": c,
                 "75% of houses are below": df[c].quantile(0.75),
                 "biggest value": df.loc[i, c],
                 "its GrLivArea": df.loc[i, "GrLivArea"],
                 "its SalePrice": df.loc[i, "SalePrice"],
                 "is one of the 2 flagged houses": bool(outlier_rule.loc[i])})
pd.DataFrame(rows).set_index("column")

**Observed**
- The trend is clear: bigger house, higher price.
- Two houses (red) are way off: 4,676 and 5,642 sq ft but sold for only 184,750 and 160,000, even though both have the top quality score (10). Both are `Partial` sales in `Edwards`. `Partial` means the house was not finished when it was assessed, so the price probably does not match the size.
- Two other very big houses (about 4,300-4,500 sq ft, about 750k) follow the trend, so we keep them. That is why the rule needs both "big area" and "low price".
- After removing the two flagged houses, the correlation between area and price goes up. So those two houses were pulling the relationship down.
- The other big values in the last table are rare. Unlike the two flagged houses, none of them comes with a price that clearly breaks the size pattern.

**Action**
- Remove the 2 flagged houses (0.14% of the data).
- Keep the other big values. The log step in the pipeline (2.4) makes them less extreme.

### 1.7. Columns near constant

In [ ]:
top_share = pd.Series({c: df[c].value_counts(dropna=False, normalize=True).iloc[0] for c in df.columns.drop("SalePrice")})

near_const_report = top_share[top_share >= 0.99].sort_values(ascending=False).to_frame("pct share common value")
near_const_report["most common value"] = [("NA (no pool)" if pd.isna(df[c].value_counts(dropna=False).index[0])
                                          else df[c].value_counts(dropna=False).index[0]) for c in near_const_report.index]
display(near_const_report)

**Observed**: Four columns have the same value in at least 99.5% of houses: `Utilities` (99.9% `AllPub`), `Street` (99.6% `Pave`), `PoolArea` and `PoolQC` (99.5%, only 7 houses have a pool).

**Action**: drop these four columns ('Street', 'Utilities', 'PoolArea', 'PoolQC'). We only looked at how often values repeat, never at the price, so nothing leaks.


### 1.8. One more column to drop: `GarageYrBlt`

`GarageYrBlt` is year garage was built

In [ ]:
has_garage = df["GarageYrBlt"].notna()
print("Houses with no garage year:", int((~has_garage).sum()))
print("Correlation of GarageYrBlt with YearBuilt (houses that have a garage):",
      round(df.loc[has_garage, "GarageYrBlt"].corr(df.loc[has_garage, "YearBuilt"]), 3))

**Observed**
- 81 houses have no garage year (they have no garage).
- For houses with a garage, the garage year is very close to the year the house was built.

**Action**: drop `GarageYrBlt`. It mostly repeats `YearBuilt`, and filling the 81 gaps would mean inventing a year. `GarageType = None` still tells us there is no garage.


### 1.9. Apply the cleaning

Everything above in one function. The function can be reused later on Kaggle's `test.csv` (use `drop_outliers=False` there, because we never remove rows from a test file).

What it does:
1. Remove the 2 flagged houses (only when `drop_outliers=True`).
2. Drop the near-constant columns and `GarageYrBlt`.
3. Fill the "none" columns with `None` (text) or `0` (`MasVnrArea`).
4. Turn `MSSubClass` and `MoSold` into text.
5. Turn the rating letters into 0-5.
6. Leave `LotFrontage` and `Electrical` empty for the pipeline.

In [ ]:
rating_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
near_constant = ["Street", "Utilities", "PoolArea", "PoolQC"]   # from 1.7
drop_cols = near_constant + ["GarageYrBlt"]


def clean_data(data: pd.DataFrame, drop_outliers: bool = False) -> pd.DataFrame:
    out = data.drop(columns=["Id"], errors="ignore").copy()

    if drop_outliers:
        out = out.loc[~((out["GrLivArea"] > OUT_AREA) & (out["SalePrice"] < OUT_PRICE))]

    out = out.drop(columns=drop_cols, errors="ignore")

    none_cols = [c for c in absent_cat_cols if c in out.columns] + ["MasVnrType"]
    out[none_cols] = out[none_cols].fillna("None")
    out["MasVnrArea"] = out["MasVnrArea"].fillna(0)

    out["MSSubClass"] = out["MSSubClass"].astype(str)
    out["MoSold"] = out["MoSold"].astype(str)

    for c in rating_cols:
        if c in out.columns:
            out[c] = out[c].map(rating_map)

    text_cols = out.select_dtypes(exclude="number").columns
    out[text_cols] = out[text_cols].apply(lambda s: s.str.strip())
    return out


df_clean = clean_data(df, drop_outliers=True)

mapped = [c for c in rating_cols if c in df_clean.columns]
left = df_clean.isna().sum()
print("Rows, columns:", df.shape, "->", df_clean.shape)
print("Rating columns turned into numbers:", len(mapped), mapped)
print("Rating values that didn't match the map:", int(df_clean[mapped].isna().sum().sum()))
print()
print("Still empty:")
print(left[left > 0].to_string())

Before and after, for three houses that have no garage:

In [ ]:
demo_idx = df.index[df["GarageType"].isna()][:3]
demo_cols = ["GarageType", "GarageQual", "BsmtQual", "FireplaceQu", "MasVnrType", "ExterQual"]
print("RAW:")
display(df.loc[demo_idx, demo_cols])
print("CLEAN:")
display(df_clean.loc[demo_idx, demo_cols])

- 1,460 rows and 80 columns became 1,458 rows and 75 columns (2 flagged houses, 4 near-constant columns and `GarageYrBlt` are gone).
- Every rating letter was matched. Nothing turned into a new missing value.
- Only `LotFrontage` (259) and `Electrical` (1) are still empty. Those are the two really-missing columns from 1.2.
- The before/after table shows the changes: `NaN` became `None` where the house has no garage, basement or fireplace, and `ExterQual` letters became numbers (for example `TA` became 3).


## 2. DATA PREPARATION

**Goal**: turn the cleaned table into numbers a model can use, without leaking anything from the validation set.

**Order**: split first, then build new columns, then fit one pipeline on the training set only.


### 2.1. Split into train and validation

We split 80/20. Price is split into 5 equal groups and each group is shared between train and validation, so both sets have cheap and expensive houses.

We call the 20% part validation, not test, because our's own `test.csv` has no prices.

We also create `log1p(SalePrice)`. The reason is in 3.1.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns="SalePrice")
y = df_clean["SalePrice"]

price_group = pd.qcut(y, q=5, labels=False)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=price_group)

y_train_log, y_valid_log = np.log1p(y_train), np.log1p(y_valid)

print("Train:", X_train.shape, "| Validation:", X_valid.shape)
pd.DataFrame({"train": y_train.describe(), "validation": y_valid.describe()}).round(0)

**Observed**
- 1,166 houses to train, 292 to validate.
- The quartiles are close: 130,000 / 163,000 / 214,000 in train vs 128,500 / 161,500 / 214,925 in validation. The median is 163,000 vs 161,500.
- But the validation mean is lower (175,937 vs 182,184, about 3% less), and the validation set has no very expensive houses (max 538,000 vs 755,000 in train, and a smaller spread).

**Action**: the split is fair for most houses, but validation is a bit "easier" at the top end. Keep this in mind when reading scores. Cross-validation on the train set later will give a steadier number.


### 2.2. New columns

We combine related columns into one, so the model sees the idea directly:
- `TotalSF` = basement + 1st floor + 2nd floor area
- `TotalBath` = full baths + half of the half baths (above ground and basement)
- `HouseAge` = year sold minus year built
- `RemodAge` = year sold minus year remodelled

Are they better than the columns they come from? Correlation with log price (train only). We also test a `TotalPorchSF` (all porches added up) to see if it helps.


In [ ]:
def add_features(d: pd.DataFrame) -> pd.DataFrame:
    out = d.copy()
    out["TotalSF"] = out["TotalBsmtSF"] + out["1stFlrSF"] + out["2ndFlrSF"]
    out["TotalBath"] = out["FullBath"] + 0.5 * out["HalfBath"] + out["BsmtFullBath"] + 0.5 * out["BsmtHalfBath"]
    out["HouseAge"] = (out["YrSold"] - out["YearBuilt"]).clip(lower=0)     # clip: never below 0
    out["RemodAge"] = (out["YrSold"] - out["YearRemodAdd"]).clip(lower=0)
    return out

X_train_fe = add_features(X_train)
X_valid_fe = add_features(X_valid)

porch_total = X_train[["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]].sum(axis=1)   # tested only

compare = [
    ("TotalSF (new)",      X_train_fe["TotalSF"]),
    ("GrLivArea",          X_train_fe["GrLivArea"]),
    ("TotalBsmtSF",        X_train_fe["TotalBsmtSF"]),
    ("1stFlrSF",           X_train_fe["1stFlrSF"]),
    ("TotalBath (new)",    X_train_fe["TotalBath"]),
    ("FullBath",           X_train_fe["FullBath"]),
    ("HalfBath",           X_train_fe["HalfBath"]),
    ("HouseAge (new)",     X_train_fe["HouseAge"]),
    ("YearBuilt",          X_train_fe["YearBuilt"]),
    ("RemodAge (new)",     X_train_fe["RemodAge"]),
    ("YearRemodAdd",       X_train_fe["YearRemodAdd"]),
    ("TotalPorchSF (tested)", porch_total),
    ("OpenPorchSF",        X_train_fe["OpenPorchSF"]),
]
display(pd.DataFrame({"correlation with log price": {name: s.corr(y_train_log) for name, s in compare}}))

**Observed**
- `TotalSF` (0.831) beats every piece it is made of: `GrLivArea` 0.739, `TotalBsmtSF` 0.649, `1stFlrSF` 0.629.
- `TotalBath` (0.678) beats `FullBath` (0.601) and `HalfBath` (0.321).
- `HouseAge` (-0.576) and `YearBuilt` (0.575) have the same strength with opposite signs. `RemodAge` (-0.574) and `YearRemodAdd` (0.571) too. They look like the same information.
- `TotalPorchSF` (0.193) is weaker than `OpenPorchSF` alone (0.318).

**Action**
- Keep `TotalSF`, `TotalBath`, `HouseAge` and `RemodAge`.
- Do not build `TotalPorchSF` (adding the porches together made the signal weaker).
- `HouseAge` and `RemodAge` are easier to read than raw years. Whether the year columns stay is decided in 2.3.


### 2.3. Drop columns that copy another column

If two columns move almost exactly together, the model gets the same information twice. We list every pair with a correlation of 0.95 or more (in absolute value) in the train set.


In [ ]:
num_fe = X_train_fe.select_dtypes(include="number")
corr_fe = num_fe.corr()

pairs = (corr_fe.where(np.triu(np.ones(corr_fe.shape, dtype=bool), k=1))
                .stack().rename("correlation").reset_index())
pairs.columns = ["column 1", "column 2", "correlation"]
copy_pairs = pairs[pairs["correlation"].abs() >= 0.95].sort_values("correlation", key=abs, ascending=False)
display(copy_pairs.reset_index(drop=True))

copycat_drop = ["YearBuilt", "YearRemodAdd", "GarageCond"]
X_train_fe = X_train_fe.drop(columns=copycat_drop)
X_valid_fe = X_valid_fe.drop(columns=copycat_drop)
print("Dropped:", copycat_drop)
print("Train shape now:", X_train_fe.shape)

**Observed**: three pairs are almost identical: `YearBuilt` vs `HouseAge` (-0.999), `YearRemodAdd` vs `RemodAge` (-0.998), and `GarageQual` vs `GarageCond` (0.954).

**Action**
- Drop `YearBuilt` and `YearRemodAdd`.
- Drop `GarageCond` and keep `GarageQual`. 
- Less strong overlaps (0.8 to 0.9) are shown in 3.2.

### 2.4. The pipeline

We have three kinds of columns. Each kind gets its own steps. Every number the steps learn (medians, skew, scaling, categories) comes from the train set only.


In [ ]:
area_cols = ["LotFrontage", "LotArea", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
             "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", "GarageArea", "WoodDeckSF", "OpenPorchSF",
             "EnclosedPorch", "3SsnPorch", "ScreenPorch", "MiscVal", "TotalSF"]

continuous_cols = [c for c in area_cols if c in X_train_fe.columns]
other_num_cols = [c for c in X_train_fe.select_dtypes(include="number").columns if c not in continuous_cols]
cat_cols = X_train_fe.select_dtypes(exclude="number").columns.tolist()

pd.DataFrame({
    "group": ["Areas and money (can be any size)", "Other numbers (counts, years, ratings)", "Text columns"],
    "how many": [len(continuous_cols), len(other_num_cols), len(cat_cols)],
    "examples": [", ".join(continuous_cols[:4]), ", ".join(other_num_cols[:4]), ", ".join(cat_cols[:4])],
    "steps": ["fill gaps with median -> log (only if it helps) -> scale",
              "fill gaps with median -> scale",
              "fill gaps with most common value -> one-hot (rare categories grouped)"],
}).set_index("group")

Text columns become one-hot columns (one 0/1 column per category). How many would that be?

In [ ]:
n_categories = X_train_fe[cat_cols].nunique().sort_values(ascending=False)
rare = sum(int((X_train_fe[c].value_counts() < 10).sum()) for c in cat_cols)

print(f"{len(cat_cols)} text columns, {int(n_categories.sum())} categories in total")
print(f"Categories with fewer than 10 houses in train: {rare}")
print()
n_categories.head(8).to_frame("number of categories")

**Observed**
- 33 text columns would turn into the number of one-hot columns shown above, for only 1,166 training houses. Many categories are rare (the count of categories with fewer than 10 houses is printed above).
- Earlier we also saw that 3 of 25 neighbourhoods have fewer than 10 houses (3.5).

**Action**
- Group every category with fewer than 10 houses into one "other" column per text column (`min_frequency=10`). The model can't learn anything stable from 2 or 3 houses.
- Fill `LotFrontage` with the median of the **same neighbourhood**, because lot frontage differs a lot between areas (2.5 shows the numbers).
- Use log only on area columns that are skewed **and** get better with log (see 2.5). A plain "log everything skewed" rule can make some columns worse.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


class NeighborhoodMedianImputer(BaseEstimator, TransformerMixin):
    """Fill empty `column` with the median of the same `group` (learned on train)."""
    def __init__(self, column="LotFrontage", group="Neighborhood"):
        self.column = column
        self.group = group

    def fit(self, X, y=None):
        self.group_median_ = X.groupby(self.group)[self.column].median()
        self.global_median_ = X[self.column].median()
        return self

    def transform(self, X):
        X = X.copy()
        fill = X[self.group].map(self.group_median_).fillna(self.global_median_)
        X[self.column] = X[self.column].fillna(fill)
        return X


class SmartLog1p(BaseEstimator, TransformerMixin):
    """log1p only on columns that are skewed (> threshold) AND less skewed after the log (checked on train)."""
    def __init__(self, threshold=0.75):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        before = X.skew()
        after = np.log1p(X.clip(lower=0)).skew()
        skewed = before > self.threshold
        better = after.abs() < before.abs()
        self.skew_before_ = before.values
        self.skew_after_ = after.values
        self.log_idx_ = np.where((skewed & better).values)[0]
        self.skipped_idx_ = np.where((skewed & ~better).values)[0]
        return self

    def transform(self, X):
        X = np.array(X, dtype=float, copy=True)
        X[:, self.log_idx_] = np.log1p(np.clip(X[:, self.log_idx_], 0, None))
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


continuous_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", SmartLog1p(0.75)),
    ("scaler", StandardScaler()),
])
other_num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10, sparse_output=False)),
])

column_transformer = ColumnTransformer([
    ("continuous", continuous_pipe, continuous_cols),
    ("other_num", other_num_pipe, other_num_cols),
    ("cat", cat_pipe, cat_cols),
])

preprocessor = Pipeline([
    ("lot_frontage", NeighborhoodMedianImputer("LotFrontage", "Neighborhood")),
    ("columns", column_transformer),
])
print("Pipeline ready")

### 2.5. Run the pipeline (fit on train, then apply to validation)

In [ ]:
X_train_prepared = preprocessor.fit_transform(X_train_fe)
X_valid_prepared = preprocessor.transform(X_valid_fe)

feature_names = preprocessor.named_steps["columns"].get_feature_names_out()
X_train_prepared_df = pd.DataFrame(X_train_prepared, columns=feature_names, index=X_train_fe.index)
X_valid_prepared_df = pd.DataFrame(X_valid_prepared, columns=feature_names, index=X_valid_fe.index)

n_onehot = sum(name.startswith("cat__") for name in feature_names)
print("Prepared train:     ", X_train_prepared_df.shape)
print("Prepared validation:", X_valid_prepared_df.shape)
print("Empty cells left (train / validation):", int(X_train_prepared_df.isna().sum().sum()), "/", int(X_valid_prepared_df.isna().sum().sum()))
print(f"One-hot columns: {n_onehot} (would be {int(n_categories.sum())} without grouping rare categories)")
print()

lot_by_nb = X_train_fe.groupby("Neighborhood")["LotFrontage"].median()
print(f"LotFrontage empty in train: {int(X_train_fe['LotFrontage'].isna().sum())}")
print(f"Median LotFrontage overall: {X_train_fe['LotFrontage'].median():.0f} | per neighbourhood: from {lot_by_nb.min():.0f} to {lot_by_nb.max():.0f}")


Which area columns got the log, and which did not?

In [ ]:
log_step = preprocessor.named_steps["columns"].named_transformers_["continuous"].named_steps["log"]

decision = np.full(len(continuous_cols), "not skewed enough", dtype=object)
decision[log_step.log_idx_] = "log applied"
decision[log_step.skipped_idx_] = "skipped (log does not help)"

skew_table = pd.DataFrame({"skew before": log_step.skew_before_,
                           "skew after log": log_step.skew_after_,
                           "decision": decision}, index=continuous_cols).sort_values("skew before", ascending=False)
skew_table

A plain example: `LotArea` before and after the log.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
lot = X_train_fe["LotArea"]
axes[0].hist(lot, bins=60, color="lightgray")
axes[0].set_title(f"LotArea (raw), skew = {lot.skew():.1f}")
axes[1].hist(np.log1p(lot), bins=60, color="steelblue")
axes[1].set_title(f"LotArea after log, skew = {np.log1p(lot).skew():.2f}")
plt.tight_layout()
plt.show()

And what a few rows look like before and after the pipeline:

In [ ]:
demo_cols = ["LotArea", "GrLivArea", "OverallQual", "ExterQual"]
prep_names = [("continuous__" if c in continuous_cols else "other_num__") + c for c in demo_cols]

print("Before (cleaned values):")
display(X_train_fe[demo_cols].head(3))
print("After (scaled numbers, mean 0 and spread 1):")
display(X_train_prepared_df[prep_names].head(3).rename(columns=dict(zip(prep_names, demo_cols))))

nb_cols = [c for c in feature_names if c.startswith("cat__Neighborhood_")]
print("Text to one-hot, first 3 houses:")
pd.DataFrame({"Neighborhood": X_train_fe["Neighborhood"].head(3),
              "column switched on": X_train_prepared_df[nb_cols].head(3).idxmax(axis=1)})

**Observed**
- Train and validation have the same columns, and there is no empty cell left anywhere.
- Grouping rare categories makes the one-hot part smaller (compare the two numbers above).
- `LotFrontage`: 209 empty cells in train. The overall median is 69, but the neighbourhood medians go from 21 to 93, so a single overall value would be wrong for many houses.
- Log works for most area columns. `LotArea` goes from skew 13.8 to -0.27 (the picture shows the long tail gone). `GrLivArea` goes from 1.04 to -0.07 and `1stFlrSF` from 0.95 to 0.03.
- `BsmtUnfSF` is skipped: its skew would go from 0.91 to -2.11, which is worse. That is what the "only if it helps" rule is for.
- Some columns stay skewed even after the log: `MiscVal` 22.2 to 5.4, `3SsnPorch` 10.0 to 7.6, `LowQualFinSF` 8.6 to 7.1, `ScreenPorch` 4.2 to 3.1. Most houses have 0 in these columns, so the log can't fix the big pile at 0.
- The before/after table shows the idea: raw areas become small numbers centred on 0, and each house's neighbourhood becomes one switched-on column.

**Action**
- Use this prepared data for modelling.
- Accept that the mostly-zero columns stay skewed. Tree models don't mind. For linear models, a simple "has this feature or not" column would be the next idea to try.


## 3. EDA

We look only at the **train set**, after cleaning and the new columns, but before scaling and log, so numbers are still in normal units (sq ft, USD).


In [ ]:
eda_df = X_train_fe.copy()
eda_df["SalePrice"] = y_train
eda_df["LogSalePrice"] = y_train_log
print(eda_df.shape)

### 3.1. The price itself

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(eda_df["SalePrice"], bins=50, kde=True, ax=axes[0], color="seagreen")
axes[0].set_title("SalePrice (raw)")
sns.histplot(eda_df["LogSalePrice"], bins=50, kde=True, ax=axes[1], color="steelblue")
axes[1].set_title("log1p(SalePrice)")
plt.tight_layout()
plt.show()

print(f"Skewness  raw: {eda_df['SalePrice'].skew():.2f} | after log: {eda_df['LogSalePrice'].skew():.2f}")
print(f"Kurtosis  raw: {eda_df['SalePrice'].kurt():.2f} | after log: {eda_df['LogSalePrice'].kurt():.2f}")


(Skewness: 0 means balanced, bigger than 0 means a long tail to the right. Kurtosis: bigger means more extreme values far from the middle.)

**Observed**
- Raw price has a long tail to the right (skewness 1.97, kurtosis 6.71): most houses are around 100k-200k and a few go up to 755,000.
- After the log, the shape is close to balanced (skewness 0.23, kurtosis 0.78).

**Action**: model `log1p(SalePrice)`, and turn predictions back with `expm1`. The log also stops a few very expensive houses from dominating the errors.


### 3.2. What goes with price? (numbers)

In [ ]:
num_eda = eda_df.select_dtypes(include="number").drop(columns="SalePrice")
corr_target = num_eda.corr()["LogSalePrice"].drop("LogSalePrice").sort_values(ascending=False)

top = corr_target.abs().sort_values(ascending=False).head(15).index
plt.figure(figsize=(8, 5.5))
sns.barplot(x=corr_target[top].values, y=top, color="steelblue")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Correlation with log price")
plt.title("Top 15 numeric columns")
plt.tight_layout()
plt.show()

print(corr_target.loc[top].round(3).to_string())

In [ ]:
top10 = corr_target.abs().sort_values(ascending=False).head(10).index.tolist()
corr_top = num_eda[top10 + ["LogSalePrice"]].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_top, mask=np.triu(np.ones_like(corr_top, dtype=bool), k=1), cmap="coolwarm", center=0,
            vmin=-1, vmax=1, annot=True, fmt=".2f", annot_kws={"size": 8})
plt.title("How the top 10 relate to each other and to price")
plt.tight_layout()
plt.show()

Columns that overlap a lot with each other (0.8 or more), not counting price:

In [ ]:
corr_feat = num_eda.drop(columns="LogSalePrice").corr()
pairs = (corr_feat.where(np.triu(np.ones(corr_feat.shape, dtype=bool), k=1))
                  .stack().rename("correlation").reset_index())
pairs.columns = ["column 1", "column 2", "correlation"]
pairs[pairs["correlation"].abs() >= 0.8].sort_values("correlation", key=abs, ascending=False).reset_index(drop=True)


**Observed**
- Top of the list: `TotalSF` 0.831, `OverallQual` 0.830, `GrLivArea` 0.739. Then `ExterQual` 0.690, `GarageCars` 0.687, `KitchenQual` 0.683, `TotalBath` 0.678, `GarageArea` 0.655, `TotalBsmtSF` 0.649, `1stFlrSF` 0.629.
- So price goes with three things: **how big** (`TotalSF`, `GrLivArea`), **how well built** (`OverallQual` and the rating columns), and **garage and bathrooms**.
- `HouseAge` is -0.576: newer houses sell for more.
- Overlapping columns (the last table): `GarageCars` and `GarageArea` 0.894, `GrLivArea` and `TotalSF` 0.870, `Fireplaces` and `FireplaceQu` 0.867, `GrLivArea` and `TotRmsAbvGrd` 0.829, `TotalBsmtSF` with `TotalSF` and with `1stFlrSF` 0.813.
- Correlation only sees straight-line links. Text columns like `Neighborhood` are not in this chart at all (see 3.5 and 3.6).

**Action**
- Keep all of these columns for now. The overlaps are not close to a copy (we already dropped those in 2.3).
- When modelling: tree models don't mind overlap; for linear models, use a method that handles it (Ridge or Lasso).


### 3.3. Size and quality vs price

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

sns.regplot(data=eda_df, x="GrLivArea", y="LogSalePrice", scatter_kws={"s": 12, "alpha": 0.4}, line_kws={"color": "red"}, ax=axes[0])
axes[0].set_title("GrLivArea vs log price")

sns.regplot(data=eda_df, x="TotalSF", y="LogSalePrice", scatter_kws={"s": 12, "alpha": 0.4}, line_kws={"color": "red"}, ax=axes[1])
axes[1].set_title("TotalSF vs log price")

sns.boxplot(data=eda_df, x="OverallQual", y="SalePrice", color="lightsteelblue", ax=axes[2])
axes[2].set_title("OverallQual vs price")

plt.tight_layout()
plt.show()

eda_df.groupby("OverallQual")["SalePrice"].agg(["count", "median"]).T

**Observed**
- Both area charts go up in a nearly straight line. The `TotalSF` points sit closer to the red line than the `GrLivArea` points, which matches the correlations (0.831 vs 0.739).
- `OverallQual`: the median price goes up at every level, from 50,150 (level 1) to 445,365 (level 10). The jumps get bigger at the top: 199,400 (7), 270,500 (8), 374,000 (9).
- The boxes also get taller with quality, so expensive houses vary more in price.
- Levels 1, 2 and 10 have very few houses (2, 3 and 16), so those medians are less sure.

**Action**: size and quality both matter a lot, and `TotalSF` is the better size column. The price grows faster than a straight line in dollars, which is another reason to use the log price.


### 3.4. Rating columns vs price

In [ ]:
rating_names = {0: "None", 1: "Po", 2: "Fa", 3: "TA", 4: "Gd", 5: "Ex"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, c in zip(axes, ["ExterQual", "KitchenQual"]):
    g = eda_df.groupby(c)["SalePrice"].agg(["count", "median"])
    labels = [rating_names[i] for i in g.index]
    ax.bar(labels, g["median"], color="steelblue")
    for x, (n, m) in enumerate(zip(g["count"], g["median"])):
        ax.text(x, m, f"n={n}", ha="center", va="bottom", fontsize=9)
    ax.set_title(f"Median price by {c}")
    ax.set_ylabel("Median SalePrice")
plt.tight_layout()
plt.show()

**Observed**: for both `ExterQual` and `KitchenQual`, every step up the rating (Fa, TA, Gd, Ex) gives a higher median price. Most houses sit in `TA` and `Gd`, and `Ex` has only a few houses (see the `n=` labels).

**Action**: turning the letters into 0-5 numbers (1.4) was a good move. The model sees the order directly.


### 3.5. Location

`Neighborhood` is the part of Ames the house is in. The names used below: `NridgHt` = Northridge Heights, `NoRidge` = Northridge, `StoneBr` = Stone Brook, `BrDale` = Briardale, `IDOTRR` = near the Iowa DOT and railroad, `MeadowV` = Meadow Village.


In [ ]:
order = eda_df.groupby("Neighborhood")["SalePrice"].median().sort_values(ascending=False).index

plt.figure(figsize=(13, 5))
sns.boxplot(data=eda_df, x="Neighborhood", y="SalePrice", order=order, color="lightsteelblue")
plt.xticks(rotation=60)
plt.title("Price by neighbourhood (sorted by median)")
plt.tight_layout()
plt.show()

nb_table = eda_df.groupby("Neighborhood").agg(houses=("SalePrice", "count"),
                                              median_price=("SalePrice", "median"),
                                              median_OverallQual=("OverallQual", "median")).sort_values("median_price", ascending=False)
print("Top 3 and bottom 3 neighbourhoods:")
display(pd.concat([nb_table.head(3), nb_table.tail(3)]))
print("Neighbourhoods with fewer than 10 houses in train:", int((nb_table["houses"] < 10).sum()), "of", len(nb_table))


**Observed**
- Location has a big effect. The median price in the most expensive area (`NridgHt`, 318,061) is about 3.7 times the cheapest (`MeadowV`, 86,000). Top 3: `NridgHt`, `NoRidge`, `StoneBr`. Bottom 3: `BrDale`, `IDOTRR`, `MeadowV`.
- The expensive areas also have a higher median `OverallQual` than the cheap areas (last column). So part of the price gap is better-built houses, not only the address.
- 3 of the 25 neighbourhoods have fewer than 10 houses.

**Action**: keep `Neighborhood` as a column (it can't show up in the correlation chart because it is text). The rare areas are grouped by the pipeline (2.4).


### 3.6. Sale type and zoning

`SaleCondition`: `Normal` = normal sale, `Partial` = house was not finished when assessed (mostly new homes), `Abnorml` = foreclosure, short sale or trade, `Family` = sold between family, `Alloca` = two linked properties, `AdjLand` = neighbouring land.

`MSZoning`: `RL` = residential low density, `RM` = residential medium density, `RH` = residential high density, `FV` = floating village residential, `C (all)` = commercial.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, col in zip(axes, ["SaleCondition", "MSZoning"]):
    ordr = eda_df.groupby(col)["SalePrice"].median().sort_values(ascending=False).index
    sns.boxplot(data=eda_df, x=col, y="SalePrice", order=ordr, color="lightsteelblue", ax=ax)
    ax.set_title(f"Price by {col}")
plt.tight_layout()
plt.show()

sale_table = eda_df.groupby("SaleCondition").agg(houses=("SalePrice", "count"),
                                                 median_price=("SalePrice", "median"),
                                                 median_HouseAge=("HouseAge", "median")).sort_values("median_price", ascending=False)
zone_table = eda_df.groupby("MSZoning")["SalePrice"].agg(["count", "median"]).sort_values("median", ascending=False)
display(sale_table)
display(zone_table)

**Observed**
- `Partial` sales are the most expensive: median 253,293 vs 159,975 for `Normal` (about 58% more). The `median_HouseAge` column shows `Partial` houses are very new.
- `Abnorml` sales (foreclosure and similar) are cheaper: 138,150, about 14% below `Normal`.
- Zoning: `FV` 202,925 and `RL` 173,000 are the highest, `RM` 120,250 is lower, and `C (all)` is the lowest at 81,000 (only 9 houses).

**Action**: keep both columns. Part of the `Partial` effect is simply "new house", so expect `SaleCondition` and `HouseAge` to share some of the work.


### 3.7. Age and time of sale

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.regplot(data=eda_df, x="HouseAge", y="LogSalePrice", scatter_kws={"s": 12, "alpha": 0.4}, line_kws={"color": "red"}, ax=axes[0])
axes[0].set_title("HouseAge vs log price")

sns.boxplot(data=eda_df, x="YrSold", y="SalePrice", color="lightsteelblue", ax=axes[1])
axes[1].set_title("Price by year sold")

month_order = [str(m) for m in range(1, 13)]
sns.countplot(data=eda_df, x="MoSold", order=month_order, color="lightsteelblue", ax=axes[2])
axes[2].set_title("Number of sales per month")

plt.tight_layout()
plt.show()

print("Median price by year sold:")
print(eda_df.groupby("YrSold")["SalePrice"].median().to_string())

**Observed**
- `HouseAge`: price drops fast for the first ~40 years, then flattens (the points bend, the red straight line only shows the average trend). Brand-new houses (age 0) range from cheap to the most expensive in the data.
- `YrSold`: median price goes 164,500 (2006), 168,000, 165,500, 160,200, 154,650 (2010). That is only about 6% lower from 2006 to 2010, and the boxes overlap almost completely.
- `MoSold`: sales have a clear season. June has the most (about 200), February the fewest (about 40). Month-to-month medians in 1.4 do not follow that pattern.

**Action**
- Keep `HouseAge`: it matters, but the wide spread says quality and size matter more.
- `YrSold` has little effect on price, so it is a weak column. We keep it only because `HouseAge` was built from it.
- Keep `MoSold` as text (a label), as decided in 1.4.

### 3.8. Summary

| # | What we found | Evidence | What we do |
|---|---|---|---|
| 1 | Price has a long tail to the right | skewness 1.97 raw vs 0.23 after log (3.1) | Model `log1p(SalePrice)` |
| 2 | Size and quality drive price | `TotalSF` 0.831, `OverallQual` 0.830, `GrLivArea` 0.739 (3.2, 3.3) | Keep. `TotalSF` is a helpful new column |
| 3 | Rating letters are a real scale | price rises with each step in `ExterQual` and `KitchenQual` (3.4) | Keep them as 0-5 numbers |
| 4 | Location matters a lot | `NridgHt` 318,061 vs `MeadowV` 86,000, about 3.7 times (3.5) | Keep `Neighborhood`, group rare areas |
| 5 | Part of the location gap is quality | expensive areas have higher median `OverallQual` (3.5) | Models may share the effect between the two columns, that's OK |
| 6 | `Partial` sales cost more | 253,293 vs 159,975, about 58% more, and these houses are very new (3.6) | Keep `SaleCondition`, expect overlap with `HouseAge` |
| 7 | Newer houses cost more, with a wide spread | `HouseAge` -0.576 (3.2, 3.7) | Keep `HouseAge` |
| 8 | Year sold barely matters | median only about 6% lower from 2006 to 2010 (3.7) | `YrSold` is weak. Keep `MoSold` as text |
| 9 | Some columns overlap | `GarageCars`/`GarageArea` 0.894, `GrLivArea`/`TotalSF` 0.870 and more (3.2) | Keep for now. Use Ridge/Lasso for linear models |


## 4. DEEP LEARNING WITH PYTORCH

**Goal**: train a neural network on the prepared table from section 2, check how well it does on the validation set, and explain which columns drive its predictions.

**What we reuse from earlier sections**
- `X_train_prepared_df`, `X_valid_prepared_df`: the scaled, one-hot encoded features from the pipeline (2.5). The pipeline was fitted on the train set only.
- `y_train_log`, `y_valid_log`: the target `log1p(SalePrice)` (2.1). Section 3.1 showed the log scale removes most of the right skew (skewness 1.97 → 0.23).

Because the network learns on the log scale, its MSE is the squared error of `log(price)`. That roughly measures **relative** (percentage) error, which is also how Kaggle scores this competition (RMSE of the log price).

| Step | What | Why |
|---|---|---|
| 4.1 | Custom `Dataset`/`DataLoader`, residual MLP, picks the device on its own | Loads batches efficiently and gives a stable deep model |
| 4.2 | AdamW, LR scheduler, gradient clipping, early stopping, checkpointing | Training that can be repeated and is protected against overfitting |
| 4.3 | Metrics in USD (RMSE, MAE, $R^2$), baselines, plots | Honest results on data the model never saw |
| 4.4 | SHAP (expected gradients) | Explains *why* the model predicts what it does |

In [ ]:
%pip install -q torch shap

### 4.1. PyTorch architecture and data handling

#### 4.1.1. Reproducibility, device selection and an inner validation split

**Device-agnostic code.** `get_device()` picks the fastest backend it finds: NVIDIA GPU (`cuda`), then Apple Silicon GPU (`mps`), then `cpu`. All tensors and the model are moved with `.to(DEVICE)`, so the same notebook runs on any machine without changes. Everything stays in `float32`, because MPS does not support `float64`.

**Reproducibility.** `set_seed()` fixes the Python, NumPy and PyTorch random generators and turns on deterministic cuDNN kernels, so a run can be repeated. (Some GPU kernels, mostly on MPS, are still not bit-for-bit deterministic, so expect tiny differences between machines.)

**Why an extra split?** Early stopping, the LR scheduler and checkpointing all choose things based on a validation loss. If we used the 2.1 validation set for that, the model would be *selected on* the same houses it is later *scored on*, and the reported error would be too optimistic. So we split the training set again: **85% fit set** (gradient updates) and **15% early-stopping set** (model selection). The 292 validation houses are touched **once**, in 4.3. The split is stratified on price quintiles, as in 2.1.

In [ ]:
import copy
import random
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


def set_seed(seed: int = RANDOM_STATE) -> None:
    """Fix every random number generator used in the notebook.

    Args:
        seed: The seed shared by Python, NumPy and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    """Return the best available accelerator.

    The priority order is CUDA (NVIDIA GPU), then MPS (Apple Silicon GPU),
    then CPU.

    Returns:
        The ``torch.device`` that tensors and the model should live on.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps_backend = getattr(torch.backends, "mps", None)
    if mps_backend is not None and mps_backend.is_available():
        return torch.device("mps")
    return torch.device("cpu")


set_seed()
DEVICE = get_device()
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

# Inner split: 85% to fit the weights, 15% for early stopping.
price_group_train = pd.qcut(y_train_log, q=5, labels=False)
X_fit_df, X_es_df, y_fit_log, y_es_log = train_test_split(
    X_train_prepared_df, y_train_log, test_size=0.15,
    random_state=RANDOM_STATE, stratify=price_group_train)

pd.DataFrame(
    {"rows": [len(X_fit_df), len(X_es_df), len(X_valid_prepared_df)],
     "role": ["update the weights",
              "early stopping, LR scheduler, checkpoint",
              "final scoring only (4.3)"]},
    index=["fit set", "early-stopping set", "validation set"])

#### 4.1.2. Custom `Dataset` and `DataLoader`

`TabularDataset` turns the features into one `float32` tensor **once**, when it is created. `__getitem__` then only indexes that tensor, so pandas is never called inside the training loop. The target gets the shape `(N, 1)` to match the network output and avoid silent broadcasting in `MSELoss` (a `(N,)` vs `(N, 1)` mismatch would give an `(N, N)` loss matrix).

`make_dataloaders` builds the loaders:
- **Training loader**: shuffled every epoch with a seeded `torch.Generator` (reproducible). `drop_last=True` drops a tiny last batch: a batch with a single row would make `BatchNorm1d` fail, and a very small one gives noisy statistics.
- **Evaluation loaders**: not shuffled and a larger batch size (no gradients are kept, so memory is not an issue).
- `pin_memory` is only turned on for CUDA, where it speeds up host-to-GPU copies. `num_workers=0` because the whole dataset fits in memory, and worker processes cause problems in macOS notebooks.

In [ ]:
class TabularDataset(Dataset):
    """In-memory PyTorch dataset for a dense tabular design matrix.

    The data is converted to ``float32`` tensors once at construction time,
    so ``__getitem__`` is a cheap tensor indexing operation.

    Attributes:
        features: Tensor of shape ``(n_samples, n_features)``.
        target: Tensor of shape ``(n_samples, 1)`` or ``None`` (inference).
    """

    def __init__(self, features, target=None) -> None:
        """Build the dataset.

        Args:
            features: Array-like of shape ``(n_samples, n_features)``
                (NumPy array or pandas DataFrame).
            target: Optional array-like of shape ``(n_samples,)``. Leave it
                as ``None`` for inference-only datasets.

        Raises:
            ValueError: If features and target have different lengths.
        """
        self.features = torch.as_tensor(
            np.asarray(features, dtype=np.float32))
        self.target = None
        if target is not None:
            self.target = torch.as_tensor(
                np.asarray(target, dtype=np.float32)).reshape(-1, 1)
            if len(self.target) != len(self.features):
                raise ValueError("features and target lengths differ.")

    def __len__(self) -> int:
        """Return the number of samples."""
        return self.features.shape[0]

    def __getitem__(self, idx):
        """Return one sample.

        Args:
            idx: Integer index of the sample.

        Returns:
            ``(x, y)`` if a target is available, otherwise only ``x``.
        """
        if self.target is None:
            return self.features[idx]
        return self.features[idx], self.target[idx]


def make_dataloaders(X_fit, y_fit, X_eval, y_eval, batch_size=64,
                     eval_batch_size=256, device=DEVICE,
                     seed=RANDOM_STATE):
    """Create the training and evaluation ``DataLoader`` objects.

    Args:
        X_fit: Training features.
        y_fit: Training target (log scale).
        X_eval: Evaluation (early-stopping) features.
        y_eval: Evaluation target (log scale).
        batch_size: Mini-batch size used for gradient updates.
        eval_batch_size: Batch size for evaluation (no gradients kept).
        device: Target device. Only used to decide ``pin_memory``.
        seed: Seed for the shuffling generator.

    Returns:
        A tuple ``(train_loader, eval_loader)``.
    """
    generator = torch.Generator().manual_seed(seed)
    pin = device.type == "cuda"
    train_loader = DataLoader(
        TabularDataset(X_fit, y_fit), batch_size=batch_size, shuffle=True,
        drop_last=True, generator=generator, num_workers=0,
        pin_memory=pin)
    eval_loader = DataLoader(
        TabularDataset(X_eval, y_eval), batch_size=eval_batch_size,
        shuffle=False, num_workers=0, pin_memory=pin)
    return train_loader, eval_loader


BATCH_SIZE = 64
train_loader, es_loader = make_dataloaders(
    X_fit_df, y_fit_log, X_es_df, y_es_log, batch_size=BATCH_SIZE)

xb, yb = next(iter(train_loader))
print(f"Batches per epoch: {len(train_loader)} (train), "
      f"{len(es_loader)} (early stopping)")
print(f"One batch: x {tuple(xb.shape)} {xb.dtype}, "
      f"y {tuple(yb.shape)} {yb.dtype}")

#### 4.1.3. Network: a residual MLP for tabular data

A plain deep MLP is hard to train: gradients have to pass through every layer, and adding layers can make the training error *worse* (the degradation problem). We use **residual connections** (He et al., 2016), adapted to tabular data as in the ResNet baseline of Gorishniy et al. (2021, *Revisiting Deep Learning Models for Tabular Data*):

```
input (n_features)
  └─ Linear → BatchNorm → LeakyReLU → Dropout              (input projection to hidden_dim)
      └─ ResidualBlock × n_blocks:   h = h + F(h)
           F(h) = BN → LeakyReLU → Linear → BN → LeakyReLU → Dropout → Linear
          └─ BN → LeakyReLU → Linear(hidden_dim → 1)         (regression head: log price)
```

| Choice | Reason |
|---|---|
| **Skip connection** `h + F(h)` | The identity path lets gradients flow straight back, so deeper nets train well. Each block only has to learn a *correction* to its input. |
| **Pre-activation order** (BN → act → Linear inside `F`) | He et al. (2016b): keeps the skip path a pure identity, which trains better than putting the activation after the sum. |
| **Batch Normalization** | Keeps each layer's inputs at a stable scale, so a higher learning rate works and the loss surface is smoother. It also adds a little regularization through batch noise. |
| **LeakyReLU** (slope 0.01) | Unlike ReLU, neurons with negative input still pass a small gradient, so they can't "die". This matters here because many inputs are sparse one-hot columns. |
| **Dropout** (p = 0.3) | We have only ~1,000 rows and ~200 features, so overfitting is the main risk. Dropout randomly switches off units, which acts like averaging many smaller networks. |
| **Kaiming init** (for LeakyReLU) | Keeps the size of the signal steady through the layers at the start of training. |
| **Zero-init of the last layer in each residual branch** | Each block starts as an exact identity (Goyal et al., 2017), so at the start the network acts like a shallow model and adds depth as it learns. |
| **Output bias = mean log price** | The first prediction is already the sensible "always guess the average" baseline (≈ 12.0), not 0. This avoids a large, unhelpful loss in the first epochs. |

In [ ]:
class ResidualBlock(nn.Module):
    """Pre-activation residual block for tabular data.

    Computes ``h + F(h)`` where ``F`` is
    ``BN -> LeakyReLU -> Linear -> BN -> LeakyReLU -> Dropout -> Linear``.

    Attributes:
        branch: The residual branch ``F`` as an ``nn.Sequential``.
    """

    def __init__(self, dim: int, dropout: float,
                 negative_slope: float = 0.01) -> None:
        """Build the block.

        Args:
            dim: Width of the hidden representation (input = output).
            dropout: Dropout probability inside the branch.
            negative_slope: Negative slope of the LeakyReLU.
        """
        super().__init__()
        self.branch = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.LeakyReLU(negative_slope),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the residual mapping.

        Args:
            x: Tensor of shape ``(batch, dim)``.

        Returns:
            Tensor of shape ``(batch, dim)``.
        """
        return x + self.branch(x)


class ResidualMLP(nn.Module):
    """Residual multilayer perceptron for tabular regression.

    Attributes:
        input_layer: Projection from ``n_features`` to ``hidden_dim``.
        blocks: Stack of ``ResidualBlock`` modules.
        head: Final normalisation and linear layer giving one output.
    """

    def __init__(self, n_features: int, hidden_dim: int = 256,
                 n_blocks: int = 3, dropout: float = 0.3,
                 negative_slope: float = 0.01,
                 target_mean: float = 0.0) -> None:
        """Build the network.

        Args:
            n_features: Number of input columns.
            hidden_dim: Width of every hidden layer.
            n_blocks: Number of residual blocks.
            dropout: Dropout probability.
            negative_slope: Negative slope of every LeakyReLU.
            target_mean: Mean of the (log) training target, used as the
                initial output bias.
        """
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(dropout),
        )
        self.blocks = nn.Sequential(*[
            ResidualBlock(hidden_dim, dropout, negative_slope)
            for _ in range(n_blocks)
        ])
        self.head = nn.Sequential(
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(negative_slope),
            nn.Linear(hidden_dim, 1),
        )
        self._init_weights(negative_slope, target_mean)

    def _init_weights(self, negative_slope: float,
                      target_mean: float) -> None:
        """Initialise the weights.

        Kaiming-normal for every linear layer, zero for the last linear
        layer of each residual branch (block starts as identity), and a
        small head whose bias equals the target mean.

        Args:
            negative_slope: LeakyReLU slope used by the Kaiming formula.
            target_mean: Initial value of the output bias.
        """
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, a=negative_slope,
                                        nonlinearity="leaky_relu")
                nn.init.zeros_(module.bias)
        for block in self.blocks:
            nn.init.zeros_(block.branch[-1].weight)
        nn.init.normal_(self.head[-1].weight, std=0.01)
        nn.init.constant_(self.head[-1].bias, target_mean)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Predict ``log1p(SalePrice)``.

        Args:
            x: Tensor of shape ``(batch, n_features)``.

        Returns:
            Tensor of shape ``(batch, 1)``.
        """
        return self.head(self.blocks(self.input_layer(x)))


MODEL_CONFIG = {
    "n_features": X_fit_df.shape[1],
    "hidden_dim": 256,
    "n_blocks": 3,
    "dropout": 0.3,
    "negative_slope": 0.01,
    "target_mean": float(y_fit_log.mean()),
}


def build_model(config: dict = MODEL_CONFIG,
                device: torch.device = DEVICE) -> ResidualMLP:
    """Create a ``ResidualMLP`` from a config dict and move it to a device.

    Args:
        config: Keyword arguments for ``ResidualMLP``.
        device: Device to place the model on.

    Returns:
        The initialised model.
    """
    return ResidualMLP(**config).to(device)


set_seed()
model = build_model()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")

with torch.no_grad():
    model.eval()
    first_pred = model(xb.to(DEVICE))
print(f"Sanity check: output shape {tuple(first_pred.shape)}, "
      f"mean initial prediction {first_pred.mean().item():.3f} "
      f"(target mean {MODEL_CONFIG['target_mean']:.3f})")

### 4.2. Training loop and MLOps practices

The training code is split into small functions that each do one job and can be tested on their own:

| Component | What it does |
|---|---|
| `train_one_epoch` | One pass over the fit set: forward → MSE loss → backward → **gradient clipping** → AdamW step. |
| `evaluate` | Loss on a loader with `model.eval()` (dropout off, BatchNorm uses running statistics) and `torch.no_grad()`. |
| `EarlyStopping` | Tracks the best early-stopping loss, **saves `best_model_weights.pth`** every time it improves, and stops after `patience` epochs with no improvement. |
| `fit` | Runs the loop, steps the scheduler, logs MSE/RMSE per epoch, and reloads the best weights at the end. |

**Optimiser: AdamW** (Loshchilov & Hutter, 2019). In plain Adam, L2 weight decay is mixed into the adaptive gradient scaling, so weights with large gradient history are barely regularized. AdamW applies the decay **directly to the weights**, separate from the gradient step, which gives proper regularization. We use `weight_decay = 1e-2`.

**Scheduler: `ReduceLROnPlateau`** halves the learning rate when the early-stopping loss has not improved for 10 epochs. A large LR at the start makes fast progress, and smaller steps later settle into a good minimum. We prefer it to a fixed cosine schedule because we don't know in advance how many epochs early stopping will allow.

**Gradient clipping** (`clip_grad_norm_`, max norm 1.0). If the total gradient norm goes above the limit, the whole gradient is scaled down, keeping its direction. This protects against a single bad batch (for example one with an unusual house) causing a huge update.

**Metrics.** We log MSE (the loss) and RMSE = $\sqrt{\text{MSE}}$ on the log scale every epoch. Because $\log(\hat{y}) - \log(y) \approx (\hat{y}-y)/y$ for small errors, an RMSE of 0.12 means a typical error of about 12% of the price.

> Note: the *training* loss is averaged while dropout is on, so it is a slightly pessimistic estimate. That is why the early-stopping loss can sit below the training loss early in training. It is normal and not a sign of leakage.

In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader,
                    criterion: nn.Module,
                    optimizer: torch.optim.Optimizer,
                    device: torch.device,
                    max_grad_norm: float = 1.0) -> float:
    """Run one optimisation epoch.

    Args:
        model: The network to train.
        loader: Training ``DataLoader`` yielding ``(x, y)`` batches.
        criterion: Loss function (``nn.MSELoss``).
        optimizer: Optimiser updating ``model``'s parameters.
        device: Device holding the model.
        max_grad_norm: Maximum global L2 norm of the gradients.

    Returns:
        Sample-weighted mean training loss (MSE) of the epoch.
    """
    model.train()
    total_loss, n_seen = 0.0, 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        n_seen += xb.size(0)
    return total_loss / n_seen


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module,
             device: torch.device) -> float:
    """Compute the mean loss on a loader without updating the model.

    Args:
        model: The network to evaluate.
        loader: ``DataLoader`` yielding ``(x, y)`` batches.
        criterion: Loss function (``nn.MSELoss``).
        device: Device holding the model.

    Returns:
        Sample-weighted mean loss (MSE).
    """
    model.eval()
    total_loss, n_seen = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        total_loss += criterion(model(xb), yb).item() * xb.size(0)
        n_seen += xb.size(0)
    return total_loss / n_seen


class EarlyStopping:
    """Stop training when the monitored loss stops improving.

    The model weights are written to ``checkpoint_path`` every time the
    loss reaches a new best value.

    Attributes:
        patience: Epochs without improvement allowed before stopping.
        min_delta: Minimum decrease that counts as an improvement.
        checkpoint_path: Where the best ``state_dict`` is saved.
        best_loss: Best loss seen so far.
        best_epoch: Epoch (1-based) at which ``best_loss`` was reached.
        counter: Epochs since the last improvement.
    """

    def __init__(self, patience: int = 40, min_delta: float = 1e-5,
                 checkpoint_path="best_model_weights.pth") -> None:
        """Create the callback.

        Args:
            patience: Epochs without improvement allowed before stopping.
            min_delta: Minimum decrease that counts as an improvement.
            checkpoint_path: File path of the saved best weights.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.checkpoint_path = Path(checkpoint_path)
        self.best_loss = float("inf")
        self.best_epoch = 0
        self.counter = 0

    def step(self, loss: float, model: nn.Module, epoch: int) -> bool:
        """Update the state with the loss of the current epoch.

        Args:
            loss: Monitored (early-stopping) loss of this epoch.
            model: Model whose weights are saved on improvement.
            epoch: Current epoch number (1-based).

        Returns:
            ``True`` if the loss improved (and a checkpoint was written).
        """
        if loss < self.best_loss - self.min_delta:
            self.best_loss, self.best_epoch, self.counter = loss, epoch, 0
            self.checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), self.checkpoint_path)
            return True
        self.counter += 1
        return False

    @property
    def should_stop(self) -> bool:
        """Whether the patience budget is used up."""
        return self.counter >= self.patience


def fit(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
        device: torch.device, epochs: int = 500, lr: float = 1e-3,
        weight_decay: float = 1e-2, max_grad_norm: float = 1.0,
        patience: int = 40, scheduler_patience: int = 10,
        scheduler_factor: float = 0.5,
        checkpoint_path="best_model_weights.pth",
        log_every: int = 10) -> pd.DataFrame:
    """Train a model with AdamW, LR scheduling and early stopping.

    After training, the best checkpoint is loaded back into ``model``.

    Args:
        model: The network to train (already on ``device``).
        train_loader: Loader for gradient updates.
        val_loader: Loader for early stopping and the LR scheduler.
        device: Device holding the model.
        epochs: Maximum number of epochs.
        lr: Initial learning rate.
        weight_decay: Decoupled weight decay of AdamW.
        max_grad_norm: Gradient clipping threshold (global L2 norm).
        patience: Early-stopping patience in epochs.
        scheduler_patience: Epochs without improvement before the LR is
            reduced.
        scheduler_factor: Factor that multiplies the LR on a plateau.
        checkpoint_path: File path of the best weights.
        log_every: Print a log line every ``log_every`` epochs (use 1 to
            print every epoch). All epochs are always kept in the history.

    Returns:
        DataFrame with one row per epoch: train/val MSE, train/val RMSE,
        learning rate and a flag for checkpointed epochs.
    """
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                  weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=scheduler_factor,
        patience=scheduler_patience, min_lr=1e-6)
    stopper = EarlyStopping(patience, checkpoint_path=checkpoint_path)

    history = []
    start = time.perf_counter()
    print(f"{'epoch':>5} | {'train MSE':>9} | {'val MSE':>9} | "
          f"{'train RMSE':>10} | {'val RMSE':>9} | {'lr':>8}")
    for epoch in range(1, epochs + 1):
        lr_now = optimizer.param_groups[0]["lr"]
        train_mse = train_one_epoch(model, train_loader, criterion,
                                    optimizer, device, max_grad_norm)
        val_mse = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_mse)
        improved = stopper.step(val_mse, model, epoch)

        history.append({"epoch": epoch,
                        "train_mse": train_mse, "val_mse": val_mse,
                        "train_rmse": np.sqrt(train_mse),
                        "val_rmse": np.sqrt(val_mse),
                        "lr": lr_now, "checkpoint": improved})

        if epoch % log_every == 0 or epoch == 1 or stopper.should_stop:
            mark = " *" if improved else ""
            print(f"{epoch:>5} | {train_mse:>9.5f} | {val_mse:>9.5f} | "
                  f"{np.sqrt(train_mse):>10.5f} | {np.sqrt(val_mse):>9.5f}"
                  f" | {lr_now:>8.1e}{mark}")
        if stopper.should_stop:
            print(f"Early stopping: no improvement for {patience} epochs.")
            break

    elapsed = time.perf_counter() - start
    model.load_state_dict(torch.load(checkpoint_path, map_location=device,
                                     weights_only=True))
    print(f"\nFinished in {elapsed:.1f}s. Best epoch {stopper.best_epoch}: "
          f"val MSE {stopper.best_loss:.5f}, "
          f"val RMSE {np.sqrt(stopper.best_loss):.5f}. "
          f"Weights saved to '{checkpoint_path}'.")
    return pd.DataFrame(history).set_index("epoch")

Run the training. All settings sit in one dictionary, so the experiment is easy to report and repeat. (`*` marks an epoch where a new best checkpoint was saved.)

In [ ]:
TRAIN_CONFIG = {
    "epochs": 500,
    "lr": 1e-3,
    "weight_decay": 1e-2,
    "max_grad_norm": 1.0,
    "patience": 40,
    "scheduler_patience": 10,
    "scheduler_factor": 0.5,
    "checkpoint_path": "best_model_weights.pth",
    "log_every": 10,
}

set_seed()
model = build_model()
history = fit(model, train_loader, es_loader, DEVICE, **TRAIN_CONFIG)

best_epoch = int(history["val_mse"].idxmin())
display(history.loc[[1, best_epoch, history.index[-1]]]
        .rename(index={1: "first", best_epoch: "best",
                       history.index[-1]: "last"}))

### 4.3. Evaluation on the validation set

We now use the 292 validation houses for the first and only time.

1. **Load the checkpoint into a new model.** This shows that the saved `best_model_weights.pth` alone is enough to reproduce the predictions (the deployment path), independent of what is left in memory.
2. **Predict on the log scale, then go back to dollars** with the inverse of `log1p`: $\hat{y}_{\text{USD}} = \operatorname{expm1}(\hat{y}_{\log}) = e^{\hat{y}_{\log}} - 1$.
3. **Metrics in USD**:
   - $\text{RMSE} = \sqrt{\tfrac{1}{n}\sum_i (y_i - \hat{y}_i)^2}$: typical error in dollars. Big misses count extra because the errors are squared.
   - $\text{MAE} = \tfrac{1}{n}\sum_i |y_i - \hat{y}_i|$: average absolute error in dollars. Less affected by a few big misses.
   - $R^2 = 1 - \tfrac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$: share of price variance explained (1 = perfect, 0 = no better than the mean).
   - We also report **RMSE on the log scale** (the Kaggle metric) and **MAPE** (average % error), because errors on house prices are best judged relative to the price.

To put the numbers in context, we compare against two **baselines** on exactly the same features: a model that always predicts the mean (the floor), and **Ridge regression** (a regularized linear model and a strong reference for this dataset). A deep model is only worth its complexity if it is competitive with these.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


@torch.no_grad()
def predict(model: nn.Module, features, device: torch.device = DEVICE,
            batch_size: int = 256) -> np.ndarray:
    """Predict ``log1p(SalePrice)`` for a feature matrix.

    Args:
        model: Trained network.
        features: Array-like of shape ``(n_samples, n_features)``.
        device: Device holding the model.
        batch_size: Inference batch size.

    Returns:
        1-D NumPy array of log-scale predictions.
    """
    model.eval()
    loader = DataLoader(TabularDataset(features), batch_size=batch_size,
                        shuffle=False)
    preds = [model(xb.to(device)).cpu() for xb in loader]
    return torch.cat(preds).numpy().ravel()


def regression_report(y_true_log, y_pred_log) -> dict:
    """Compute regression metrics on both the log and the USD scale.

    Args:
        y_true_log: True ``log1p(SalePrice)`` values.
        y_pred_log: Predicted ``log1p(SalePrice)`` values.

    Returns:
        Dict with RMSE, MAE, R2 and MAPE in USD, and RMSE on the log scale.
    """
    y_true_log = np.asarray(y_true_log, dtype=float)
    y_pred_log = np.asarray(y_pred_log, dtype=float)
    y_true, y_pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    return {
        "RMSE (USD)": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE (USD)": mean_absolute_error(y_true, y_pred),
        "R2 (USD)": r2_score(y_true, y_pred),
        "MAPE (%)": np.mean(np.abs(y_pred - y_true) / y_true) * 100,
        "RMSE (log)": np.sqrt(mean_squared_error(y_true_log, y_pred_log)),
    }


# 1. Reload the best checkpoint into a fresh model.
best_model = build_model()
best_model.load_state_dict(torch.load(TRAIN_CONFIG["checkpoint_path"],
                                      map_location=DEVICE,
                                      weights_only=True))
best_model.eval()

# 2. Predict on the validation set and invert log1p.
y_valid_pred_log = predict(best_model, X_valid_prepared_df)
y_valid_pred = np.expm1(y_valid_pred_log)
y_valid_true = np.expm1(y_valid_log.to_numpy())

# 3. Baselines on the same features (log target).
mean_pred_log = np.full(len(y_valid_log), y_train_log.mean())
ridge = RidgeCV(alphas=np.logspace(-2, 3, 50))
ridge.fit(X_train_prepared_df, y_train_log)
ridge_pred_log = ridge.predict(X_valid_prepared_df)

results = pd.DataFrame({
    "Mean predictor": regression_report(y_valid_log, mean_pred_log),
    f"Ridge (alpha={ridge.alpha_:.1f})": regression_report(
        y_valid_log, ridge_pred_log),
    "Residual MLP (PyTorch)": regression_report(
        y_valid_log, y_valid_pred_log),
}).T
display(results.style.format({
    "RMSE (USD)": "{:,.0f}", "MAE (USD)": "{:,.0f}", "R2 (USD)": "{:.4f}",
    "MAPE (%)": "{:.2f}", "RMSE (log)": "{:.4f}"}))

#### 4.3.1. Plots

- **Learning curves**: training vs early-stopping loss (MSE and RMSE, log scale) per epoch. The dashed line marks the restored best epoch, and dotted lines mark each learning-rate reduction.
- **Actual vs predicted** (USD): each dot is a validation house, and the red dashed line $y=\hat{y}$ is a perfect fit. Next to it, the **residuals** plot shows whether errors grow with price (heteroscedasticity).

Figures are also saved as 300-dpi PNGs in `figures/` so they can go straight into the report.

In [ ]:
from matplotlib.ticker import FuncFormatter

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
usd_fmt = FuncFormatter(lambda v, _: f"\\${v / 1e3:,.0f}k")


def plot_learning_curves(history: pd.DataFrame, best_epoch: int,
                         save_path=None) -> None:
    """Plot training vs validation MSE and RMSE per epoch.

    Args:
        history: Output of ``fit``.
        best_epoch: Epoch of the restored checkpoint.
        save_path: Optional file path to save the figure.
    """
    lr_drops = history.index[history["lr"].diff() < 0]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    for ax, metric in zip(axes, ["mse", "rmse"]):
        ax.plot(history.index, history[f"train_{metric}"], lw=1.8,
                label=f"Training {metric.upper()}")
        ax.plot(history.index, history[f"val_{metric}"], lw=1.8,
                label=f"Validation (early-stopping) {metric.upper()}")
        ax.axvline(best_epoch, color="black", ls="--", lw=1,
                   label=f"Best epoch ({best_epoch})")
        for i, ep in enumerate(lr_drops):
            ax.axvline(ep, color="gray", ls=":", lw=0.8,
                       label="LR reduced" if i == 0 else None)
        ax.set_yscale("log")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(f"{metric.upper()} (log1p scale)")
        ax.set_title(f"Training vs validation {metric.upper()}")
        ax.legend(frameon=True)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_actual_vs_predicted(y_true, y_pred, metrics: dict,
                             save_path=None) -> None:
    """Scatter of actual vs predicted prices plus a residual plot.

    Args:
        y_true: True prices in USD.
        y_pred: Predicted prices in USD.
        metrics: Output of ``regression_report`` (shown in the legend box).
        save_path: Optional file path to save the figure.
    """
    pct_err = np.abs(y_pred - y_true) / y_true * 100
    lims = [min(y_true.min(), y_pred.min()) * 0.95,
            max(y_true.max(), y_pred.max()) * 1.05]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    sc = axes[0].scatter(y_true, y_pred, c=pct_err, cmap="viridis_r",
                         vmin=0, vmax=30, s=22, alpha=0.8,
                         edgecolor="white", linewidth=0.3)
    axes[0].plot(lims, lims, "r--", lw=1.5, label="Perfect fit (y = x)")
    axes[0].set(xlim=lims, ylim=lims, xlabel="Actual SalePrice",
                ylabel="Predicted SalePrice",
                title="Actual vs predicted (validation set)")
    axes[0].xaxis.set_major_formatter(usd_fmt)
    axes[0].yaxis.set_major_formatter(usd_fmt)
    axes[0].set_aspect("equal")
    axes[0].legend(loc="upper left")
    text = (f"RMSE = \\${metrics['RMSE (USD)']:,.0f}\n"
            f"MAE  = \\${metrics['MAE (USD)']:,.0f}\n"
            f"$R^2$ = {metrics['R2 (USD)']:.4f}")
    axes[0].text(0.97, 0.03, text, transform=axes[0].transAxes,
                 ha="right", va="bottom", family="monospace",
                 bbox={"boxstyle": "round", "fc": "white", "alpha": 0.9})
    fig.colorbar(sc, ax=axes[0], label="Absolute % error (capped at 30)")

    residuals = y_true - y_pred
    axes[1].scatter(y_pred, residuals, s=22, alpha=0.7,
                    edgecolor="white", linewidth=0.3)
    axes[1].axhline(0, color="red", ls="--", lw=1.5)
    axes[1].set(xlabel="Predicted SalePrice",
                ylabel="Residual (actual - predicted)",
                title="Residuals vs predicted")
    axes[1].xaxis.set_major_formatter(usd_fmt)
    axes[1].yaxis.set_major_formatter(usd_fmt)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


plot_learning_curves(history, best_epoch, FIG_DIR / "learning_curves.png")
nn_metrics = results.loc["Residual MLP (PyTorch)"].to_dict()
plot_actual_vs_predicted(y_valid_true, y_valid_pred, nn_metrics,
                         FIG_DIR / "actual_vs_predicted.png")

worst = pd.DataFrame({"actual": y_valid_true, "predicted": y_valid_pred},
                     index=y_valid_log.index)
worst["abs_pct_error"] = (worst["predicted"] - worst["actual"]).abs() \
    / worst["actual"] * 100
print("Five largest relative errors:")
display(worst.join(X_valid_fe[["Neighborhood", "OverallQual", "TotalSF",
                               "SaleCondition"]])
        .nlargest(5, "abs_pct_error").round(1))

**How to read the results** (numbers from our reference run; a Colab GPU gives slightly different values, but the same picture)
- **Baselines.** The mean predictor sets the floor ($R^2 \approx 0$, RMSE ≈ \$67.6k). The residual MLP reached **RMSE ≈ \$17.4k, MAE ≈ \$12.5k, $R^2 \approx 0.933$** (log-RMSE ≈ 0.126), which is on par with the tuned Ridge model ($R^2 \approx 0.930$). This is the expected result on ~1,000 rows: the log price is close to linear in the prepared features (3.2, 3.3), so a well-regularized linear model is a very strong reference. A neural net matching it shows the pipeline is sound. A clear win would need more data to learn non-linear interactions. (Smaller networks, higher dropout or heavier weight decay all scored *worse* in our tests, so we kept this configuration.)
- **Learning curves.** The early-stopping loss reaches its minimum early (around epoch 10). After that the training loss keeps falling while the early-stopping loss levels off at a higher value, and the gap that opens is textbook overfitting. This is exactly why early stopping and checkpointing matter: the model we keep is the one from the best epoch, not the last one. The dotted lines show the scheduler halving the LR whenever progress stalls.
- **Actual vs predicted.** The points follow the $y = x$ line closely across the whole range, including the most expensive validation house (~\$538k). The residual plot shows no clear funnel in dollar terms: most errors stay within ±\$40k, with no visible trend against the predicted price, so the model is not systematically biased in any price range.
- **Largest errors.** In *relative* terms, the worst misses are very cheap houses (under ~\$85k, mostly in `OldTown` and `IDOTRR`) that the model overprices by 60–120%. These houses are rare in training and their low prices probably reflect condition problems that the features capture only partly. On the log scale, a \$40k house predicted at \$89k is a large error even though it is small in dollars.


> **Limitation.** The network was trained on 85% of the training set (the rest was used for early stopping), while Ridge used 100%. A common last step is to retrain on the full training set for `best_epoch` epochs before predicting on Kaggle's `test.csv`.

### 4.4. Model interpretability with SHAP

A neural network has no simple coefficients to read, so we use **SHAP** (SHapley Additive exPlanations; Lundberg & Lee, 2017). For every house $i$ and feature $j$, SHAP gives a value $\phi_{ij}$ such that

$$
f(x_i) \approx \mathbb{E}[f(X)] + \sum_{j} \phi_{ij},
$$

so each prediction is split **additively** into the average prediction plus one contribution per feature. Because the model predicts $\log(1+\text{price})$, $\phi_{ij}$ is a contribution in log units: $\phi = +0.10$ means the feature pushes the price up by about $e^{0.10} - 1 \approx 10.5\%$.

**Why `GradientExplainer` (expected gradients)?** It combines Integrated Gradients with SHAP's sampling from a background set (Erion et al., 2021). It only needs gradients, so it works with any differentiable PyTorch graph, including our residual additions and BatchNorm. `DeepExplainer` (DeepLIFT) needs special rules for each layer type and is known to break on non-module operations such as the residual `x + F(x)`.

**Setup choices**
- **Background**: 200 random rows from the *fit set* stand for "the average house" $\mathbb{E}[f(X)]$. The validation set is never used as background.
- **Explained rows**: all 292 validation houses (data the model never trained on).
- The model is copied to the CPU in `eval()` mode, so BatchNorm and Dropout behave in a fixed way.
- **Grouping one-hot columns.** `Neighborhood` is split into ~20 one-hot columns, so each column alone looks weak. Because SHAP values add up, we can **sum** them back to one value per original column and judge the importance of each original column fairly.

In [ ]:
import shap

N_BACKGROUND, N_SAMPLES = 200, 500

shap_model = copy.deepcopy(best_model).to("cpu").eval()
rng = np.random.default_rng(RANDOM_STATE)
bg_idx = rng.choice(len(X_fit_df), size=N_BACKGROUND, replace=False)
background = torch.tensor(X_fit_df.to_numpy()[bg_idx], dtype=torch.float32)
X_explain = torch.tensor(X_valid_prepared_df.to_numpy(), dtype=torch.float32)

explainer = shap.GradientExplainer(shap_model, background)
raw_shap = explainer.shap_values(X_explain, nsamples=N_SAMPLES,
                                 rseed=RANDOM_STATE)

# Normalise the output format across SHAP versions to (n_rows, n_features).
shap_values = np.asarray(raw_shap[0] if isinstance(raw_shap, list)
                         else raw_shap)
if shap_values.ndim == 3:
    shap_values = shap_values[..., 0]
print("SHAP matrix:", shap_values.shape)

# Sanity check of additivity: base value + sum(phi) should be close to f(x).
with torch.no_grad():
    base_value = shap_model(background).mean().item()
    f_x = shap_model(X_explain).numpy().ravel()
gap = np.abs(base_value + shap_values.sum(axis=1) - f_x)
print(f"Base value E[f(X)] = {base_value:.4f} "
      f"(about ${np.expm1(base_value):,.0f})")
print(f"Additivity check: mean |gap| = {gap.mean():.4f} log units "
      f"(prediction std = {f_x.std():.4f})")

#### 4.4.1. SHAP summary plot (one dot per house per feature)

Features are ranked by mean $|\phi|$. Each dot is one validation house: its position on the x-axis is how much that feature moved this house's predicted log price, and its colour is the feature value (red = high, blue = low, after scaling). A clean red-right / blue-left split means "higher value → higher price".

In [ ]:
def clean_feature_name(name: str) -> str:
    """Remove the ColumnTransformer prefixes from a feature name.

    Args:
        name: Name such as ``"continuous__TotalSF"`` or
            ``"cat__Neighborhood_NridgHt"``.

    Returns:
        A readable name such as ``"TotalSF"`` or ``"Neighborhood_NridgHt"``.
    """
    return name.split("__", 1)[-1]


def original_column(name: str, categorical: list) -> str:
    """Map a prepared feature name back to its original column.

    Args:
        name: Prepared feature name (with ColumnTransformer prefix).
        categorical: Original names of the one-hot encoded columns.

    Returns:
        The original column name (e.g. ``"Neighborhood"``).
    """
    short = clean_feature_name(name)
    if name.startswith("cat__"):
        matches = [c for c in categorical if short.startswith(f"{c}_")]
        if matches:
            return max(matches, key=len)
    return short


display_names = [clean_feature_name(n) for n in X_valid_prepared_df.columns]

plt.figure()
shap.summary_plot(shap_values, X_valid_prepared_df.to_numpy(),
                  feature_names=display_names, max_display=20, show=False)
plt.title("SHAP summary: residual MLP, validation set")
plt.xlabel("SHAP value (impact on predicted log1p(SalePrice))")
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_summary.png", dpi=300, bbox_inches="tight")
plt.show()

#### 4.4.2. Importance per original column (one-hot columns summed)

In [ ]:
groups = [original_column(n, cat_cols) for n in X_valid_prepared_df.columns]
shap_grouped = (pd.DataFrame(shap_values, columns=groups,
                             index=X_valid_prepared_df.index)
                .T.groupby(level=0).sum().T)

importance = pd.DataFrame({
    "mean |SHAP| (log units)": shap_grouped.abs().mean(),
    "share of total (%)": (shap_grouped.abs().mean()
                           / shap_grouped.abs().mean().sum() * 100),
}).sort_values("mean |SHAP| (log units)", ascending=False)
importance["approx. price effect (%)"] = np.expm1(
    importance["mean |SHAP| (log units)"]) * 100

top_n = 15
plt.figure(figsize=(8, 6))
sns.barplot(x=importance["mean |SHAP| (log units)"].head(top_n),
            y=importance.index[:top_n], color="steelblue")
plt.xlabel("Mean |SHAP value| (log1p scale)")
plt.ylabel("")
plt.title(f"Top {top_n} original columns (one-hot SHAP values summed)")
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_grouped_importance.png", dpi=300,
            bbox_inches="tight")
plt.show()

# Direction of the effect: rank correlation between the raw value and SHAP.
top3 = importance.index[:3].tolist()
rows = []
for col in top3:
    raw = X_valid_fe[col]
    if pd.api.types.is_numeric_dtype(raw):
        rho = raw.rank().corr(shap_grouped[col].rank())
        direction = (f"higher value -> higher price (Spearman {rho:+.2f})"
                     if rho > 0 else
                     f"higher value -> lower price (Spearman {rho:+.2f})")
    else:
        by_cat = shap_grouped[col].groupby(raw).mean().sort_values()
        direction = (f"most negative: {by_cat.index[0]} "
                     f"({by_cat.iloc[0]:+.3f}), most positive: "
                     f"{by_cat.index[-1]} ({by_cat.iloc[-1]:+.3f})")
    rows.append({"column": col, "direction": direction})

display(importance.head(10).round(4))
print("Top 3 columns and the direction of their effect:")
display(pd.DataFrame(rows).set_index("column"))

#### 4.4.3. Analysis of the three most influential features

> The numbers below come from our reference run. Exact SHAP values change slightly with hardware (CUDA/MPS/CPU) and seeds, so check them against the tables printed above. The top-3 *set* was stable across our runs.

The SHAP analysis shows that the network has learned a valuation logic consistent with both the exploratory findings of Section 3 and hedonic pricing theory, in which a dwelling's price breaks down into the implicit prices of its attributes. The single most influential feature is **`OverallQual`** (mean $|\phi| \approx 0.033$ log units, i.e. it moves a typical prediction by about 3.3%, and up to tens of percent for the lowest and highest grades). Its effect is strictly monotonic and positive: the Spearman correlation between the feature value and its SHAP value is +0.97, so every grade step adds value, matching the step-wise median increases seen in Section 3.3. The second and third drivers are both measures of **size**: **`GrLivArea`** ($\approx 0.028$) and the engineered **`TotalSF`** ($\approx 0.025$), each with an almost perfectly monotonic positive effect (Spearman +0.99). That two different size columns sit in the top three shows that the network uses complementary information: `GrLivArea` captures *above-ground* living space, which buyers value most, while `TotalSF` adds basement area. It also supports the feature-engineering decision of Section 2.2. Together, quality and size account for the largest share of attributions, which mirrors their top-ranked correlations with log price (0.830, 0.739 and 0.831 in Section 3.2). Two caveats apply. First, SHAP explains *the model*, not causal effects in the housing market. Second, the attributions are relatively evenly spread (no single feature explains more than ~5% of the total), because highly correlated features (`TotalSF` vs `GrLivArea`, `TotalBsmtSF`, `1stFlrSF`; `OverallQual` vs `ExterQual`, `KitchenQual`) *share* credit. So the importance of the underlying **concepts** of size and quality is more reliable than the exact rank of any single column within them.

### 4.5. Summary of the deep learning section

| Aspect | What we did | Why it matters |
|---|---|---|
| Data handling | `TabularDataset` (tensors built once), seeded shuffling, `drop_last` for BatchNorm | Fast, reproducible batching |
| Architecture | Pre-activation residual MLP with BatchNorm, LeakyReLU, Dropout, zero-init residual branches | Stable optimisation and regularization on a small tabular dataset |
| Portability | `get_device()`: CUDA → MPS → CPU, all `float32` | The same code runs on any machine |
| Optimisation | AdamW (decoupled weight decay), `ReduceLROnPlateau`, gradient clipping | Well-regularized, stable training |
| Model selection | Early stopping and checkpointing on a separate **inner** validation split | The validation score stays unbiased (no selection leakage) |
| Evaluation | `expm1` back to USD; RMSE, MAE, $R^2$, MAPE, log-RMSE; mean and Ridge baselines | Honest metrics with context |
| Interpretability | SHAP expected gradients, additivity check, one-hot grouping | Shows the model's logic matches domain knowledge |

**References**
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR*. / Identity mappings in deep residual networks. *ECCV*.
- Gorishniy, Y., Rubachev, I., Khrulkov, V., & Babenko, A. (2021). Revisiting deep learning models for tabular data. *NeurIPS*.
- Goyal, P., et al. (2017). Accurate, large minibatch SGD: Training ImageNet in 1 hour. *arXiv:1706.02677*.
- Ioffe, S., & Szegedy, C. (2015). Batch normalization. *ICML*.
- Srivastava, N., et al. (2014). Dropout: A simple way to prevent neural networks from overfitting. *JMLR*.
- Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*.
- Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS*.
- Erion, G., Janizek, J. D., Sturmfels, P., Lundberg, S. M., & Lee, S.-I. (2021). Improving performance of deep learning models with axiomatic attribution priors and expected gradients. *Nature Machine Intelligence*.